# 🌟 Star Schema Creation - WORKING VERSION
## Shoebadoo Sales Analytics - Phase 3

---

## 1. Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, year, month, quarter, dayofmonth, dayofweek, weekofyear,
    date_format, when, monotonically_increasing_id, row_number, to_date
)
from pyspark.sql.window import Window
import warnings
warnings.filterwarnings('ignore')

spark = SparkSession.builder \
    .appName("Shoebadoo_StarSchema") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print("✅ Spark Session created")
print(f"Spark Version: {spark.version}")

## 2. Cleaned Data laden

In [ ]:
INPUT_PATH = "/app/data/cleaned/2_data_cleaning"
OUTPUT_PATH = "/app/data/warehouse/3_star_schema"

sales_df = spark.read.parquet(f"{INPUT_PATH}/sales_clean.parquet")
products_df = spark.read.parquet(f"{INPUT_PATH}/products_clean.parquet")
customers_df = spark.read.parquet(f"{INPUT_PATH}/customers_clean.parquet")

print(f"✅ Sales: {sales_df.count():,} rows")
print(f"✅ Products: {products_df.count():,} rows")
print(f"✅ Customers: {customers_df.count():,} rows")
print("\n📋 Sales Schema:")
sales_df.printSchema()

## 3. DIM_DATE erstellen

**WICHTIG:** Die Spalte heißt `sale_datetime`, nicht `sale_date`!

In [ ]:
# Date aus sale_datetime extrahieren
dates_df = sales_df.select(
    to_date(col("sale_datetime")).alias("full_date")
).distinct()

# Date Attributes
dim_date = dates_df.select(
    date_format(col("full_date"), "yyyyMMdd").cast("int").alias("date_key"),
    col("full_date"),
    year("full_date").alias("year"),
    quarter("full_date").alias("quarter"),
    month("full_date").alias("month"),
    date_format("full_date", "MMMM").alias("month_name"),
    weekofyear("full_date").alias("week"),
    dayofmonth("full_date").alias("day"),
    dayofweek("full_date").alias("day_of_week"),
    date_format("full_date", "EEEE").alias("day_name"),
    when(dayofweek("full_date").isin([1, 7]), True).otherwise(False).alias("is_weekend")
).orderBy("date_key")

print(f"✅ dim_date: {dim_date.count():,} rows")
dim_date.show(5)

## 4. DIM_CUSTOMER erstellen

In [ ]:
window = Window.orderBy("customer_id")

dim_customer = customers_df.select(
    row_number().over(window).alias("customer_key"),
    col("customer_id"),
    col("customer_name"),
    col("customer_segment"),
    col("country"),
    col("city"),
    col("postal_code"),
    col("registration_date")
)

print(f"✅ dim_customer: {dim_customer.count():,} rows")
dim_customer.show(5)

## 5. DIM_PRODUCT erstellen

In [ ]:
window = Window.orderBy("product_id")

dim_product = products_df.select(
    row_number().over(window).alias("product_key"),
    col("product_id"),
    col("product_name"),
    col("category"),
    col("subcategory"),
    col("brand"),
    col("price"),
    col("color"),
    col("size")
)

print(f"✅ dim_product: {dim_product.count():,} rows")
dim_product.show(5)

## 6. DIM_CHANNEL erstellen

In [ ]:
channels_df = sales_df.select(col("channel").alias("channel_name")).distinct()
window = Window.orderBy("channel_name")

dim_channel = channels_df.select(
    row_number().over(window).alias("channel_key"),
    monotonically_increasing_id().cast("string").alias("channel_id"),
    col("channel_name"),
    when(col("channel_name") == "Online", "Digital").otherwise("Physical").alias("channel_type")
)

print(f"✅ dim_channel: {dim_channel.count():,} rows")
dim_channel.show()

## 7. FACT_SALES erstellen

In [ ]:
# Sale Date als separate Spalte
sales_with_date = sales_df.withColumn("sale_date", to_date(col("sale_datetime")))

# Date Key joinen
fact = sales_with_date.join(
    dim_date.select("date_key", col("full_date").alias("join_date")),
    sales_with_date.sale_date == col("join_date"),
    "left"
)

# Customer Key joinen
fact = fact.join(
    dim_customer.select("customer_key", col("customer_id").alias("cust_id")),
    fact.customer_id == col("cust_id"),
    "left"
)

# Product Key joinen
fact = fact.join(
    dim_product.select("product_key", col("product_id").alias("prod_id")),
    fact.product_id == col("prod_id"),
    "left"
)

# Channel Key joinen
fact = fact.join(
    dim_channel.select("channel_key", col("channel_name").alias("ch_name")),
    fact.channel == col("ch_name"),
    "left"
)

# Finale Fact Table
fact_sales = fact.select(
    col("sale_id"),
    col("date_key"),
    col("customer_key"),
    col("product_key"),
    col("channel_key"),
    col("quantity"),
    (col("total_amount") / col("quantity")).alias("unit_price"),
    col("total_amount"),
    when(col("payment_method") == "Returned", True).otherwise(False).alias("is_returned")
)

print(f"✅ fact_sales: {fact_sales.count():,} rows")
fact_sales.show(5)

## 8. Quality Check

In [ ]:
print("\n🔍 NULL CHECK:")
print("=" * 40)
checks = [
    ("date_key", fact_sales.filter(col("date_key").isNull()).count()),
    ("customer_key", fact_sales.filter(col("customer_key").isNull()).count()),
    ("product_key", fact_sales.filter(col("product_key").isNull()).count()),
    ("channel_key", fact_sales.filter(col("channel_key").isNull()).count())
]

for col_name, null_count in checks:
    status = "✅" if null_count == 0 else "❌"
    print(f"{status} {col_name}: {null_count} NULLs")
print("=" * 40)

## 9. Star Schema speichern

In [ ]:
dim_date.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_date")
dim_customer.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_customer")
dim_product.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_product")
dim_channel.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_channel")
fact_sales.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/fact_sales")

print("\n✅ STAR SCHEMA SAVED!")
print("="*50)
print(f"📁 Location: {OUTPUT_PATH}")
print("="*50)

## 10. Test Query

In [ ]:
# Umsatz nach Kanal
fact = spark.read.parquet(f"{OUTPUT_PATH}/fact_sales")
dim_ch = spark.read.parquet(f"{OUTPUT_PATH}/dim_channel")

result = fact \
    .join(dim_ch, "channel_key") \
    .groupBy("channel_name") \
    .agg({"total_amount": "sum", "quantity": "sum"}) \
    .orderBy(col("sum(total_amount)").desc())

print("\n📊 UMSATZ NACH KANAL:")
result.show()

---

## ✅ FERTIG!

**Star Schema erfolgreich erstellt!**

---